# Baseline-модель для Breast Cancer Dataset

Цель этапа - построить базовую модель Logistic Regression без масштабирования признаков и зафиксировать стартовое качество.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from experiment_utils import (
    calculate_binary_metrics,
    load_breast_cancer_data,
    save_figure,
    save_results,
    split_breast_cancer_features_target,
)

POSITIVE_LABEL = "malignant"
TEST_SIZE = 0.2
RANDOM_STATE = 42

def positive_proba(model, X, positive_label=POSITIVE_LABEL):
    classifier = model.named_steps["classifier"]
    positive_index = list(classifier.classes_).index(positive_label)
    return model.predict_proba(X)[:, positive_index]

def evaluate_pipeline(name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = positive_proba(pipeline, X_test)
    metrics = calculate_binary_metrics(y_test, y_pred, y_proba, POSITIVE_LABEL)
    return metrics, pipeline, y_pred, y_proba


## Подготовка данных

Разделение на train/test выполняется до предобработки. Импутация находится внутри `Pipeline`, чтобы избежать data leakage.

In [2]:
df = load_breast_cancer_data()
X, y = split_breast_cancer_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape


((455, 30), (114, 30))

## Обучение baseline

Baseline использует `SimpleImputer(strategy="median")` и `LogisticRegression` без scaling.

In [3]:
baseline_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", LogisticRegression(max_iter=3000, solver="liblinear")),
])
metrics, model, y_pred, y_proba = evaluate_pipeline(
    "Baseline", baseline_pipeline, X_train, X_test, y_train, y_test
)
model.named_steps["classifier"].classes_


array(['benign', 'malignant'], dtype=object)

In [4]:
pd.DataFrame([metrics])


,Accuracy,F1-score,ROC-AUC
0,0.938596,0.911392,0.993717


In [5]:
print(classification_report(y_test, y_pred, digits=4))


              precision    recall  f1-score   support

      benign     0.9221    0.9861    0.9530        72
   malignant     0.9730    0.8571    0.9114        42

    accuracy                         0.9386       114
   macro avg     0.9475    0.9216    0.9322       114
weighted avg     0.9408    0.9386    0.9377       114



## Сохранение результата

ROC-AUC считается по вероятности класса `malignant`, потому что этот класс является критичным в медицинской задаче.

In [6]:
results = pd.DataFrame([{ 
    "Dataset": "Breast Cancer",
    "Experiment": "Baseline",
    "Model": "LogisticRegression",
    **metrics,
}])
results = results.round(4)
save_results(results, RESULTS_DIR / "breast_cancer_baseline_results.csv")
results


,Dataset,Experiment,Model,Accuracy,F1-score,ROC-AUC
0,Breast Cancer,Baseline,LogisticRegression,0.9386,0.9114,0.9937


## Итоговый вывод

Baseline уже показывает высокое качество, потому что признаки хорошо структурированы и напрямую описывают свойства опухоли. Дальше проверим, улучшает ли результат масштабирование.